# 04 — Logistic Regression Baseline

This notebook trains an interpretable logistic-regression baseline to predict whether a
student will answer the current interaction correctly on the first attempt. It uses the
chronological train, validation, and test files produced in notebook 03.

The workflow emphasizes probability quality, leakage-safe evaluation, and explanations
that are useful for model auditing. Coefficients describe conditional associations, not
causal effects or fixed characteristics of a student.


## Evaluation protocol

- Read candidate predictors from the feature dictionary rather than maintaining a
  separate hard-coded model list.
- Use `log1p_opportunity`, not raw `opportunity`, for this linear baseline.
- Fit preprocessing and candidate models on training data only.
- Select L2 regularization strength and any operating threshold using validation data.
- Refit the selected model on train + validation, then evaluate the test set once.
- Compare against a prevalence-only probability baseline.
- Never include identifiers, current-interaction behavior, `correct`, or `data_split` in
  the feature matrix.

The historical features follow notebook 03's sequential protocol: validation and test
rows may use outcomes from earlier interactions when those outcomes would already have
been observed at prediction time.


## Imports and notebook setup


In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from scipy.special import expit
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("default")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

RANDOM_STATE = 42

print(f"Python:       {sys.version.split()[0]}")
print(f"pandas:       {pd.__version__}")
print(f"NumPy:        {np.__version__}")


Python:       3.13.14
pandas:       2.3.3
NumPy:        2.5.2


## Project paths


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    '''Find the nearest parent directory containing pyproject.toml.'''
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml.")


PROJECT_ROOT = find_project_root()
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_DICTIONARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "data_dictionary"
    / "skill_builder_data_feature_eng_data_dictionary.csv"
)
SPLIT_DATA_PATHS = {
    split_name: PROCESSED_DATA_DIR / f"skill_builder_data_feature_eng_{split_name}.csv"
    for split_name in ("train", "validation", "test")
}

assert FEATURE_DICTIONARY_PATH.is_file(), FEATURE_DICTIONARY_PATH
for split_name, split_path in SPLIT_DATA_PATHS.items():
    assert split_path.is_file(), f"Missing {split_name} dataset: {split_path}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature dictionary: {FEATURE_DICTIONARY_PATH.relative_to(PROJECT_ROOT)}")
for split_name, split_path in SPLIT_DATA_PATHS.items():
    print(f"{split_name.title()}: {split_path.relative_to(PROJECT_ROOT)}")


Project root: W:\Workstation ExtDrive\007 Data Science\003 Data Science Projects\2026_p019 assistments_2009_2010
Feature dictionary: data\data_dictionary\skill_builder_data_feature_eng_data_dictionary.csv
Train: data\processed\skill_builder_data_feature_eng_train.csv
Validation: data\processed\skill_builder_data_feature_eng_validation.csv
Test: data\processed\skill_builder_data_feature_eng_test.csv


## Feature policy from the data dictionary

The dictionary is the source of truth for model inclusion. For logistic regression,
the transformed opportunity count is retained and its raw counterpart is removed to
avoid representing the same quantity twice. Binary and one-hot variables pass through
without scaling; continuous/count variables are median-imputed and standardized using
training-set statistics.


In [3]:
feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH, keep_default_na=False)
assert feature_dictionary["column_name"].is_unique

include_mask = feature_dictionary["model_inclusion"].str.startswith("Include")
candidate_rows = feature_dictionary.loc[include_mask].copy()

# The dictionary recommends the transformed opportunity count for linear models.
candidate_rows = candidate_rows.loc[candidate_rows["column_name"].ne("opportunity")]
feature_names = candidate_rows["column_name"].tolist()

binary_mask = (
    candidate_rows["data_type"].str.contains("binary", case=False, na=False)
    | candidate_rows["model_inclusion"].str.contains("one-hot", case=False, na=False)
)
binary_features = candidate_rows.loc[binary_mask, "column_name"].tolist()
continuous_features = candidate_rows.loc[~binary_mask, "column_name"].tolist()

for forbidden_column in {
    "user_id",
    "order_id",
    "correct",
    "data_split",
    "attempt_count",
    "hint_count",
    "bottom_hint",
}:
    assert forbidden_column not in feature_names, f"Leakage/metadata feature: {forbidden_column}"

assert "log1p_opportunity" in feature_names
assert "opportunity" not in feature_names
assert set(binary_features).isdisjoint(continuous_features)
assert set(feature_names) == set(binary_features) | set(continuous_features)

feature_policy_summary = pd.DataFrame(
    {
        "feature_block": ["Continuous/count", "Binary/one-hot", "Total"],
        "feature_count": [
            len(continuous_features),
            len(binary_features),
            len(feature_names),
        ],
        "preprocessing": [
            "Median imputation + standardization",
            "Most-frequent imputation + passthrough",
            "L2-regularized logistic regression",
        ],
    }
)
display(feature_policy_summary)


,feature_block,feature_count,preprocessing
0,Continuous/count,38,Median imputation + standardization
1,Binary/one-hot,132,Most-frequent imputation + passthrough
2,Total,170,L2-regularized logistic regression


## Load chronological train, validation, and test data


In [4]:
TARGET = "correct"
METADATA_COLUMNS = ["user_id", "order_id", "data_split"]
use_columns = [*METADATA_COLUMNS, TARGET, *feature_names]

dtype_map = {
    **{column: "int8" for column in binary_features},
    **{column: "float64" for column in continuous_features},
    TARGET: "int8",
    "user_id": "int64",
    "order_id": "int64",
}

split_frames = {
    split_name: pd.read_csv(
        split_path,
        usecols=use_columns,
        dtype=dtype_map,
        low_memory=False,
    )
    for split_name, split_path in SPLIT_DATA_PATHS.items()
}

train = split_frames["train"]
validation = split_frames["validation"]
test = split_frames["test"]

for split_name, frame in split_frames.items():
    assert len(frame) > 0
    assert frame["data_split"].eq(split_name).all()
    assert set(frame[TARGET].unique()).issubset({0, 1})
    assert frame[feature_names].columns.tolist() == feature_names
    assert not np.isinf(frame[continuous_features].to_numpy()).any()

assert train["order_id"].max() < validation["order_id"].min()
assert validation["order_id"].max() < test["order_id"].min()

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(frame),
            "students": frame["user_id"].nunique(),
            "order_id_min": frame["order_id"].min(),
            "order_id_max": frame["order_id"].max(),
            "correct_rate": frame[TARGET].mean(),
            "missing_predictor_cells": int(frame[feature_names].isna().sum().sum()),
        }
        for split_name, frame in split_frames.items()
    ]
)
display(split_summary)


,split,rows,students,order_id_min,order_id_max,correct_rate,missing_predictor_cells
0,train,181570,3005,20224180,33447607,0.6559,0
1,validation,38907,1789,33447620,38144345,0.6092,0
2,test,38909,87,38144353,38310202,0.7169,0


## Modeling helpers


In [5]:
def build_logistic_model(c_value: float) -> Pipeline:
    '''Create a leakage-safe preprocessing and logistic-regression pipeline.'''
    continuous_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    binary_pipeline = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="most_frequent"))]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("continuous", continuous_pipeline, continuous_features),
            ("binary", binary_pipeline, binary_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    classifier = LogisticRegression(
        C=c_value,
        l1_ratio=0.0,
        solver="lbfgs",
        max_iter=1_000,
        random_state=RANDOM_STATE,
    )
    return Pipeline(
        steps=[("preprocessor", preprocessor), ("classifier", classifier)]
    )


def classification_metrics(
    y_true: pd.Series | np.ndarray,
    probability: np.ndarray,
    threshold: float,
    model_name: str,
    split_name: str,
) -> dict[str, float | str]:
    '''Return discrimination, calibration, and threshold-dependent metrics.'''
    prediction = (probability >= threshold).astype("int8")
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "split": split_name,
        "model": model_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "log_loss": log_loss(y_true, probability, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "precision_correct": precision_score(y_true, prediction, zero_division=0),
        "recall_correct": recall_score(y_true, prediction, zero_division=0),
        "specificity_incorrect": tn / (tn + fp),
        "f1_correct": f1_score(y_true, prediction, zero_division=0),
    }


## Select regularization strength on validation data

`C` is the inverse of L2 regularization strength: smaller values shrink coefficients
more strongly. Validation log loss is the primary selection criterion because the
output will be interpreted as a probability. ROC AUC, average precision, and Brier
score are retained as supporting diagnostics.


In [6]:
C_GRID = (0.01, 0.1, 1.0, 10.0)
X_train = train[feature_names]
y_train = train[TARGET]
X_validation = validation[feature_names]
y_validation = validation[TARGET]

tuning_records = []
for c_value in C_GRID:
    candidate_model = build_logistic_model(c_value)
    fit_started = time.perf_counter()
    candidate_model.fit(X_train, y_train)
    validation_probability = candidate_model.predict_proba(X_validation)[:, 1]
    classifier = candidate_model.named_steps["classifier"]
    tuning_records.append(
        {
            "C": c_value,
            "validation_log_loss": log_loss(
                y_validation, validation_probability, labels=[0, 1]
            ),
            "validation_roc_auc": roc_auc_score(y_validation, validation_probability),
            "validation_average_precision": average_precision_score(
                y_validation, validation_probability
            ),
            "validation_brier_score": brier_score_loss(
                y_validation, validation_probability
            ),
            "iterations": int(classifier.n_iter_.max()),
            "fit_seconds": time.perf_counter() - fit_started,
        }
    )

tuning_results = pd.DataFrame(tuning_records).sort_values(
    ["validation_log_loss", "C"], ignore_index=True
)
BEST_C = float(tuning_results.loc[0, "C"])

display(tuning_results.style.format(precision=4))
print(f"Selected C: {BEST_C:g}")


,C,validation_log_loss,validation_roc_auc,validation_average_precision,validation_brier_score,iterations,fit_seconds
0,0.1000,0.5464,0.7690,0.8093,0.1829,205,17.9273
1,1.0000,0.5473,0.7683,0.8089,0.1832,303,21.8383
2,10.0000,0.5474,0.7684,0.8091,0.1832,284,27.3099
3,0.0100,0.5477,0.7668,0.8067,0.1836,104,12.5130


Selected C: 0.1


## Fit the validation-selection model and choose an operating threshold


In [7]:
selection_model = build_logistic_model(BEST_C)
selection_model.fit(X_train, y_train)
validation_probability = selection_model.predict_proba(X_validation)[:, 1]

threshold_records = []
for threshold in np.linspace(0.10, 0.90, 161):
    validation_prediction = (validation_probability >= threshold).astype("int8")
    threshold_records.append(
        {
            "threshold": threshold,
            "balanced_accuracy": balanced_accuracy_score(
                y_validation, validation_prediction
            ),
            "accuracy": accuracy_score(y_validation, validation_prediction),
            "precision_correct": precision_score(
                y_validation, validation_prediction, zero_division=0
            ),
            "recall_correct": recall_score(
                y_validation, validation_prediction, zero_division=0
            ),
            "f1_correct": f1_score(
                y_validation, validation_prediction, zero_division=0
            ),
        }
    )

threshold_results = pd.DataFrame(threshold_records)
threshold_results["distance_from_0_5"] = (
    threshold_results["threshold"] - 0.5
).abs()
threshold_results = threshold_results.sort_values(
    ["balanced_accuracy", "distance_from_0_5"],
    ascending=[False, True],
    ignore_index=True,
)
SELECTED_THRESHOLD = float(threshold_results.loc[0, "threshold"])

validation_prevalence_probability = np.full(len(validation), y_train.mean())
validation_performance = pd.DataFrame(
    [
        classification_metrics(
            y_validation,
            validation_probability,
            0.5,
            "Logistic regression",
            "validation",
        ),
        classification_metrics(
            y_validation,
            validation_probability,
            SELECTED_THRESHOLD,
            "Logistic regression (selected threshold)",
            "validation",
        ),
        classification_metrics(
            y_validation,
            validation_prevalence_probability,
            0.5,
            "Training-prevalence baseline",
            "validation",
        ),
    ]
)

print(f"Validation-selected threshold: {SELECTED_THRESHOLD:.3f}")
display(threshold_results.head(10).drop(columns="distance_from_0_5"))
display(validation_performance.style.format(precision=4))


Validation-selected threshold: 0.590


,threshold,balanced_accuracy,accuracy,precision_correct,recall_correct,f1_correct
0,0.5900,0.7013,0.7182,0.7635,0.7787,0.7710
1,0.6000,0.7012,0.7155,0.7664,0.7668,0.7666
2,0.5850,0.7012,0.7196,0.7617,0.7854,0.7734
3,0.5950,0.7009,0.7167,0.7645,0.7730,0.7687
4,0.5800,0.7008,0.7206,0.7600,0.7912,0.7753
5,0.6050,0.7007,0.7136,0.7676,0.7600,0.7638
6,0.5750,0.7005,0.7218,0.7581,0.7981,0.7776
7,0.6100,0.7002,0.7119,0.7688,0.7537,0.7612
8,0.5700,0.6996,0.7224,0.7559,0.8039,0.7792
9,0.6150,0.6994,0.7096,0.7700,0.7462,0.7579


,split,model,threshold,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision_correct,recall_correct,specificity_incorrect,f1_correct
0,validation,Logistic regression,0.5000,0.7690,0.8093,0.5464,0.1829,0.7292,0.6885,0.7325,0.8750,0.5020,0.7975
1,validation,Logistic regression (selected threshold),0.5900,0.7690,0.8093,0.5464,0.1829,0.7182,0.7013,0.7635,0.7787,0.6240,0.7710
2,validation,Training-prevalence baseline,0.5000,0.5000,0.6092,0.6738,0.2403,0.6092,0.5000,0.6092,1.0000,0.0000,0.7571


## Refit on train + validation and evaluate the held-out test set

After all model choices are fixed, the selected pipeline is refit on all development
data. The test set is used only below. Both the conventional 0.50 threshold and the
validation-selected threshold are reported; threshold-free probability metrics should
remain the primary model-comparison criteria.


In [8]:
development = pd.concat([train, validation], ignore_index=True)
X_development = development[feature_names]
y_development = development[TARGET]
X_test = test[feature_names]
y_test = test[TARGET]

final_model = build_logistic_model(BEST_C)
final_fit_started = time.perf_counter()
final_model.fit(X_development, y_development)
test_probability = final_model.predict_proba(X_test)[:, 1]
final_fit_seconds = time.perf_counter() - final_fit_started

test_prevalence_probability = np.full(len(test), y_development.mean())
test_performance = pd.DataFrame(
    [
        classification_metrics(
            y_test,
            test_probability,
            0.5,
            "Logistic regression",
            "test",
        ),
        classification_metrics(
            y_test,
            test_probability,
            SELECTED_THRESHOLD,
            "Logistic regression (validation-selected threshold)",
            "test",
        ),
        classification_metrics(
            y_test,
            test_prevalence_probability,
            0.5,
            "Development-prevalence baseline",
            "test",
        ),
    ]
)

performance_table = pd.concat(
    [validation_performance, test_performance], ignore_index=True
)
print(f"Final refit time: {final_fit_seconds:,.2f} seconds")
display(performance_table.style.format(precision=4))


Final refit time: 24.07 seconds


,split,model,threshold,roc_auc,average_precision,log_loss,brier_score,accuracy,balanced_accuracy,precision_correct,recall_correct,specificity_incorrect,f1_correct
0,validation,Logistic regression,0.5000,0.7690,0.8093,0.5464,0.1829,0.7292,0.6885,0.7325,0.8750,0.5020,0.7975
1,validation,Logistic regression (selected threshold),0.5900,0.7690,0.8093,0.5464,0.1829,0.7182,0.7013,0.7635,0.7787,0.6240,0.7710
2,validation,Training-prevalence baseline,0.5000,0.5000,0.6092,0.6738,0.2403,0.6092,0.5000,0.6092,1.0000,0.0000,0.7571
3,test,Logistic regression,0.5000,0.6746,0.8251,0.5565,0.1864,0.7338,0.5892,0.7584,0.9226,0.2559,0.8325
4,test,Logistic regression (validation-selected threshold),0.5900,0.6746,0.8251,0.5565,0.1864,0.7059,0.6121,0.7764,0.8284,0.3958,0.8015
5,test,Development-prevalence baseline,0.5000,0.5000,0.7169,0.6068,0.2078,0.7169,0.5000,0.7169,1.0000,0.0000,0.8351


## Performance curves and probability diagnostics


In [9]:
diagnostic_figure = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "ROC curves",
        "Precision-recall curves",
        "Calibration curves",
        "Test predicted probabilities by outcome",
    ),
)


def thin_curve(
    x_values: np.ndarray, y_values: np.ndarray, max_points: int = 1_000
) -> tuple[np.ndarray, np.ndarray]:
    '''Retain evenly spaced plotting points while leaving metrics exact.'''
    if len(x_values) <= max_points:
        return x_values, y_values
    positions = np.unique(
        np.linspace(0, len(x_values) - 1, max_points, dtype=int)
    )
    return x_values[positions], y_values[positions]


curve_inputs = [
    ("Validation selection model", y_validation, validation_probability, "#2563EB"),
    ("Final test model", y_test, test_probability, "#D97706"),
]

for label, y_true, probability, color in curve_inputs:
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, probability)
    false_positive_rate, true_positive_rate = thin_curve(
        false_positive_rate, true_positive_rate
    )
    diagnostic_figure.add_trace(
        go.Scatter(
            x=false_positive_rate,
            y=true_positive_rate,
            mode="lines",
            name=f"{label} (AUC={roc_auc_score(y_true, probability):.3f})",
            line={"color": color},
            legendgroup=label,
        ),
        row=1,
        col=1,
    )

    precision, recall, _ = precision_recall_curve(y_true, probability)
    recall, precision = thin_curve(recall, precision)
    diagnostic_figure.add_trace(
        go.Scatter(
            x=recall,
            y=precision,
            mode="lines",
            name=f"{label} (AP={average_precision_score(y_true, probability):.3f})",
            line={"color": color},
            legendgroup=label,
            showlegend=False,
        ),
        row=1,
        col=2,
    )

    observed_rate, predicted_rate = calibration_curve(
        y_true, probability, n_bins=10, strategy="quantile"
    )
    diagnostic_figure.add_trace(
        go.Scatter(
            x=predicted_rate,
            y=observed_rate,
            mode="lines+markers",
            name=label,
            line={"color": color},
            legendgroup=label,
            showlegend=False,
        ),
        row=2,
        col=1,
    )

diagnostic_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line={"color": "#6B7280", "dash": "dash"},
        name="Reference",
        showlegend=False,
    ),
    row=1,
    col=1,
)
diagnostic_figure.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line={"color": "#6B7280", "dash": "dash"},
        name="Perfect calibration",
        showlegend=False,
    ),
    row=2,
    col=1,
)

for outcome, label, color in [
    (0, "Actually incorrect", "#DC2626"),
    (1, "Actually correct", "#059669"),
]:
    histogram_counts, histogram_edges = np.histogram(
        test_probability[y_test.to_numpy() == outcome],
        bins=np.linspace(0, 1, 31),
    )
    histogram_share = histogram_counts / histogram_counts.sum()
    histogram_centers = (histogram_edges[:-1] + histogram_edges[1:]) / 2
    diagnostic_figure.add_trace(
        go.Bar(
            x=histogram_centers,
            y=histogram_share,
            width=np.diff(histogram_edges),
            opacity=0.55,
            name=label,
            marker_color=color,
        ),
        row=2,
        col=2,
    )

diagnostic_figure.update_xaxes(title_text="False positive rate", row=1, col=1)
diagnostic_figure.update_yaxes(title_text="True positive rate", row=1, col=1)
diagnostic_figure.update_xaxes(title_text="Recall", row=1, col=2)
diagnostic_figure.update_yaxes(title_text="Precision", row=1, col=2)
diagnostic_figure.update_xaxes(title_text="Mean predicted probability", row=2, col=1)
diagnostic_figure.update_yaxes(title_text="Observed correct rate", row=2, col=1)
diagnostic_figure.update_xaxes(title_text="Predicted probability of correct", row=2, col=2)
diagnostic_figure.update_yaxes(title_text="Share within outcome", row=2, col=2)
diagnostic_figure.update_layout(
    title="Logistic-regression performance diagnostics",
    template="plotly_white",
    barmode="overlay",
    height=850,
    legend={"orientation": "h", "y": -0.12},
)
diagnostic_figure.show()


## Test confusion matrix at the validation-selected threshold


In [10]:
test_prediction = (test_probability >= SELECTED_THRESHOLD).astype("int8")
test_confusion = confusion_matrix(y_test, test_prediction, labels=[0, 1])
confusion_table = pd.DataFrame(
    test_confusion,
    index=["Actual incorrect", "Actual correct"],
    columns=["Predicted incorrect", "Predicted correct"],
)
display(confusion_table)

confusion_figure = go.Figure(
    data=go.Heatmap(
        z=test_confusion,
        x=confusion_table.columns,
        y=confusion_table.index,
        text=test_confusion,
        texttemplate="%{text:,}",
        colorscale="Blues",
        showscale=False,
        hovertemplate="%{y}<br>%{x}<br>Rows: %{z:,}<extra></extra>",
    )
)
confusion_figure.update_layout(
    title=f"Test confusion matrix (threshold = {SELECTED_THRESHOLD:.3f})",
    template="plotly_white",
    width=700,
    height=500,
    xaxis_title="Model classification",
    yaxis_title="Observed outcome",
)
confusion_figure.show()


,Predicted incorrect,Predicted correct
Actual incorrect,4360,6656
Actual correct,4787,23106


## Global coefficient explainability

Continuous/count features were standardized, so their odds ratios correspond to a
one-training-standard-deviation increase. Binary and one-hot odds ratios compare a
value of 1 with 0. The ranking is conditional on all other included predictors and can
be unstable when features are correlated. It should be used to understand model
behavior—not to claim causes or immutable student traits.


In [11]:
preprocessor = final_model.named_steps["preprocessor"]
classifier = final_model.named_steps["classifier"]
model_feature_names = preprocessor.get_feature_names_out().tolist()
coefficients = classifier.coef_.ravel()

assert len(model_feature_names) == len(feature_names) == len(coefficients)

dictionary_lookup = feature_dictionary.set_index("column_name")


def readable_feature_label(feature_name: str) -> str:
    '''Attach a plain-language skill label when the dictionary provides one.'''
    plain_name = dictionary_lookup.at[feature_name, "skill_plain_language_name"]
    if feature_name.startswith("Skill_") and plain_name:
        return f"{feature_name}: {plain_name}"
    return feature_name


coefficient_table = pd.DataFrame(
    {
        "feature": model_feature_names,
        "feature_label": [readable_feature_label(name) for name in model_feature_names],
        "feature_group": [
            dictionary_lookup.at[name, "role"] for name in model_feature_names
        ],
        "coefficient": coefficients,
        "odds_ratio": np.exp(coefficients),
        "absolute_coefficient": np.abs(coefficients),
        "interpretation_unit": [
            "1 training SD increase" if name in continuous_features else "1 versus 0"
            for name in model_feature_names
        ],
    }
).sort_values("absolute_coefficient", ascending=False, ignore_index=True)

display(coefficient_table.head(30))


,feature,feature_label,feature_group,coefficient,odds_ratio,absolute_coefficient,interpretation_unit
0,Skill_292,Skill_292: Rotations,Current problem skill indicator,-1.6835,0.1857,1.6835,1 versus 0
1,Skill_53,Skill_53: Ordering Real Numbers,Current problem skill indicator,-1.4325,0.2387,1.4325,1 versus 0
2,Skill_24,Skill_24: Congruence,Current problem skill indicator,1.3390,3.8152,1.3390,1 versus 0
3,Skill_350,Skill_350: Solving Systems of Linear Equations,Current problem skill indicator,-1.3269,0.2653,1.3269,1 versus 0
4,Skill_32,Skill_32: Nets of 3D Figures,Current problem skill indicator,1.2569,3.5145,1.2569,1 versus 0
5,Skill_14,Skill_14: Mode,Current problem skill indicator,1.1062,3.0228,1.1062,1 versus 0
6,Skill_299,Skill_299: Surface Area Cylinder,Current problem skill indicator,-1.0905,0.3360,1.0905,1 versus 0
7,Skill_325,Skill_325: Write Linear Equation from Graph,Current problem skill indicator,-1.0642,0.3450,1.0642,1 versus 0
8,Skill_204,Skill_204: Percents,Current problem skill indicator,-1.0294,0.3572,1.0294,1 versus 0
9,Skill_295,Skill_295: Area Parallelogram,Current problem skill indicator,1.0214,2.7772,1.0214,1 versus 0


In [12]:
negative_coefficients = coefficient_table.nsmallest(15, "coefficient")
positive_coefficients = coefficient_table.nlargest(15, "coefficient")
coefficient_plot_data = pd.concat(
    [negative_coefficients, positive_coefficients], ignore_index=True
).sort_values("coefficient")

coefficient_figure = go.Figure(
    go.Bar(
        x=coefficient_plot_data["coefficient"],
        y=coefficient_plot_data["feature_label"],
        orientation="h",
        marker_color=np.where(
            coefficient_plot_data["coefficient"].ge(0), "#059669", "#DC2626"
        ),
        customdata=np.column_stack(
            [
                coefficient_plot_data["odds_ratio"],
                coefficient_plot_data["interpretation_unit"],
            ]
        ),
        hovertemplate=(
            "%{y}<br>Coefficient: %{x:.3f}<br>Odds ratio: %{customdata[0]:.3f}"
            "<br>Unit: %{customdata[1]}<extra></extra>"
        ),
    )
)
coefficient_figure.add_vline(x=0, line_color="#374151", line_width=1)
coefficient_figure.update_layout(
    title="Largest positive and negative logistic-regression coefficients",
    template="plotly_white",
    height=900,
    xaxis_title="Log-odds coefficient",
    yaxis_title=None,
    margin={"l": 330},
)
coefficient_figure.show()


## Local explanation for one retrospective test prediction

The example below decomposes one typical test prediction into additive log-odds
contributions. It is selected after prediction only for model auditing. A positive
contribution raises the predicted probability of a correct first attempt; a negative
contribution lowers it. This is not a causal explanation or an instructional decision.


In [13]:
representative_position = int(np.argmin(np.abs(test_probability - np.median(test_probability))))
representative_row = test.iloc[[representative_position]]
representative_probability = float(test_probability[representative_position])

transformed_row = preprocessor.transform(representative_row[feature_names])
if hasattr(transformed_row, "toarray"):
    transformed_values = transformed_row.toarray().ravel()
else:
    transformed_values = np.asarray(transformed_row).ravel()

local_contributions = coefficients * transformed_values
intercept = float(classifier.intercept_[0])
reconstructed_probability = float(expit(intercept + local_contributions.sum()))
assert np.isclose(representative_probability, reconstructed_probability)

local_explanation = pd.DataFrame(
    {
        "feature": model_feature_names,
        "feature_label": [readable_feature_label(name) for name in model_feature_names],
        "transformed_value": transformed_values,
        "coefficient": coefficients,
        "log_odds_contribution": local_contributions,
        "odds_multiplier": np.exp(local_contributions),
    }
)
local_explanation["absolute_contribution"] = local_explanation[
    "log_odds_contribution"
].abs()
local_explanation = local_explanation.sort_values(
    "absolute_contribution", ascending=False, ignore_index=True
)

local_summary = pd.DataFrame(
    {
        "user_id": representative_row["user_id"].to_numpy(),
        "order_id": representative_row["order_id"].to_numpy(),
        "observed_correct": representative_row[TARGET].to_numpy(),
        "predicted_probability_correct": [representative_probability],
        "model_classification": [
            int(representative_probability >= SELECTED_THRESHOLD)
        ],
        "validation_selected_threshold": [SELECTED_THRESHOLD],
    }
)
display(local_summary)
display(local_explanation.head(15))


,user_id,order_id,observed_correct,predicted_probability_correct,model_classification,validation_selected_threshold
0,96286,38197278,1,0.7315,1,0.5900


,feature,feature_label,transformed_value,coefficient,log_odds_contribution,odds_multiplier,absolute_contribution
0,student_prior_correct,student_prior_correct,1.4370,0.4530,0.6510,1.9174,0.6510
1,prior_interaction_count,prior_interaction_count,1.3832,-0.3512,-0.4858,0.6152,0.4858
2,Skill_67,Skill_67: Multiplication Fractions,1.0000,-0.2309,-0.2309,0.7938,0.2309
3,min_prior_skill_interaction_count,min_prior_skill_interaction_count,0.3852,0.4916,0.1894,1.2085,0.1894
4,median_prior_skill_interaction_count,median_prior_skill_interaction_count,0.3557,-0.4653,-0.1655,0.8475,0.1655
5,student_incorrect_streak,student_incorrect_streak,-0.2235,-0.7388,0.1651,1.1795,0.1651
6,min_skill_recent_accuracy_3,min_skill_recent_accuracy_3,0.4305,0.1672,0.0720,1.0746,0.0720
7,student_prior_attempts,student_prior_attempts,1.0864,-0.0621,-0.0675,0.9347,0.0675
8,answer_type_algebra,answer_type_algebra,1.0000,0.0655,0.0655,1.0676,0.0655
9,student_prior_accuracy,student_prior_accuracy,0.3070,0.2046,0.0628,1.0648,0.0628


## Interpretation and next steps

When reviewing the executed results:

1. Prefer test ROC AUC, average precision, log loss, and Brier score for overall model
   comparison; threshold metrics depend on a use-case-specific operating point.
2. Inspect calibration before presenting probabilities as estimated chances of success.
3. Treat coefficient rankings as model-behavior diagnostics. Correlated history features
   can divide or exchange weight, even under L2 regularization.
4. Compare this baseline with a nonlinear model using the exact same splits and metrics.
5. Before deployment, add subgroup and temporal stability audits, define the educational
   cost of false alerts versus missed support needs, and involve educators in threshold
   selection.

The model is intended for teacher-facing decision support. It should not automatically
determine grades, placement, or interventions, and its output is not a measure of
intelligence, motivation, disability, or general ability.
